# Build Realized Variance Features

This notebook mirrors the Phase 2 realized-variance workflow for the EPAT VRP project.
It loads the frozen processed OHLC panel, builds RV features, and optionally writes outputs and diagnostics.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from vrp.features.realized_variance import build_rv_panel
from vrp.features.feature_io import load_feature_panel, save_feature_panel, assert_required_columns
from vrp.reports.rv_diagnostics import write_rv_diagnostics

In [2]:
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_TABLE_DIR = PROJECT_ROOT / 'reports' / 'tables'
REPORT_FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'

MARKET_CONFIG = {
    'US': {
        'input_path': DATA_PROCESSED_DIR / 'us_underlying.parquet',
        'output_path': DATA_PROCESSED_DIR / 'us_rv.parquet',
        'market': 'US',
        'symbol': 'US_UNDERLYING',
    },
    'INDIA': {
        'input_path': DATA_PROCESSED_DIR / 'india_underlying.parquet',
        'output_path': DATA_PROCESSED_DIR / 'india_rv.parquet',
        'market': 'INDIA',
        'symbol': 'INDIA_UNDERLYING',
    },
}

REQUIRED_OHLC_COLUMNS = ['date', 'open', 'high', 'low', 'close']

In [3]:
market = 'US'
window = 22
annualization_periods = 252
skip_diagnostics = False

config = MARKET_CONFIG[market]
input_path = config['input_path']
output_path = config['output_path']
market_name = config['market']
symbol = config['symbol']

df = load_feature_panel(input_path, sort_by_date=True)
assert_required_columns(df, REQUIRED_OHLC_COLUMNS, panel_name=f'{market_name} underlying panel')
df.head()

,date,open,high,low,close,adj_close,volume,source,market,symbol
0,1990-01-02,353.399994,359.690002,351.980011,359.690002,359.690002,162070000,yahoo_spx,US,^GSPC
1,1990-01-03,359.690002,360.589996,357.890015,358.760010,358.760010,192330000,yahoo_spx,US,^GSPC
2,1990-01-04,358.760010,358.760010,352.890015,355.670013,355.670013,177000000,yahoo_spx,US,^GSPC
3,1990-01-05,355.670013,355.670013,351.350006,352.200012,352.200012,158530000,yahoo_spx,US,^GSPC
4,1990-01-08,352.200012,354.239990,350.540009,353.790009,353.790009,140110000,yahoo_spx,US,^GSPC


In [4]:
rv = build_rv_panel(
    df=df,
    market=market_name,
    symbol=symbol,
    window=window,
    annualization_periods=annualization_periods,
)

rv.columns.tolist()

['date',
 'market',
 'symbol',
 'log_return',
 'simple_return',
 'gap_return',
 'intraday_return',
 'rv_cc_daily',
 'rv_parkinson_daily',
 'rv_gk_daily',
 'rv_rs_daily',
 'rv_cc_22d_ann',
 'rv_parkinson_22d_ann',
 'rv_gk_22d_ann',
 'rv_rs_22d_ann',
 'rv_yz_22d_ann']

In [5]:
rv[[
    'date',
    'rv_cc_daily',
    'rv_parkinson_daily',
    'rv_gk_daily',
    'rv_rs_daily',
    f'rv_gk_{window}d_ann',
    f'rv_yz_{window}d_ann',
]].head(30)

,date,rv_cc_daily,rv_parkinson_daily,rv_gk_daily,rv_rs_daily,rv_gk_22d_ann,rv_yz_22d_ann
0,1990-01-02,NaN,0.000169,0.000115,0.000087,NaN,NaN
1,1990-01-03,6.702339e-06,0.000020,0.000026,0.000025,NaN,NaN
2,1990-01-04,7.482762e-05,0.000098,0.000107,0.000129,NaN,NaN
3,1990-01-05,9.612119e-05,0.000054,0.000038,0.000030,NaN,NaN
4,1990-01-08,2.028881e-05,0.000040,0.000047,0.000051,NaN,NaN
5,1990-01-09,1.405814e-04,0.000061,0.000029,0.000013,NaN,NaN
6,1990-01-10,4.394484e-05,0.000084,0.000100,0.000132,NaN,NaN
7,1990-01-11,1.229597e-05,0.000024,0.000028,0.000037,NaN,NaN
8,1990-01-12,6.242309e-04,0.000249,0.000104,0.000034,NaN,NaN
9,1990-01-15,7.493953e-05,0.000036,0.000021,0.000013,NaN,NaN


In [6]:
save_feature_panel(rv, output_path, index=False)

if not skip_diagnostics:
    write_rv_diagnostics(
        {market_name: rv},
        table_dir=REPORT_TABLE_DIR,
        figure_dir=REPORT_FIGURE_DIR,
        window=window,
        annualization_periods=annualization_periods,
    )

print(f'Saved RV panel to: {output_path}')
print(f'Rows: {len(rv):,}')
print(f'Primary column first valid index: {rv[f"rv_gk_{window}d_ann"].first_valid_index()}')

Saved RV panel to: c:\Users\daksh\Desktop\EPAT\EPAT_Project_VRP_Regime\data\processed\us_rv.parquet
Rows: 9,160
Primary column first valid index: 21
